# CBFKit — Safe Reinforcement Learning Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bardhh/cbfkit/blob/main/examples/gymnasium/safe_single_integrator.ipynb)

**CBFKit** turns any continuous-control Gymnasium environment into a *provably safe* one: every action from your policy is projected by a Control Barrier Function (CBF) quadratic program before it reaches the simulator. No retraining, no reward shaping — the safety filter is a drop-in wrapper.

This notebook runs the **same** naive "drive straight at the goal" policy twice — once raw, once wrapped in a CBF safety filter — and plots the difference.

In [ ]:
# Install CBFKit (with the Gymnasium extra) straight from GitHub.
%pip install "cbfkit[gymnasium] @ git+https://github.com/bardhh/cbfkit.git"

## 1. Set up the environment and a (deliberately naive) policy

`CBFKit/SafeSingleIntegratorObstacles-v0` is a 2-D point robot that must reach a goal without hitting circular obstacles. Our policy just drives straight at the goal — it knows nothing about the obstacles.

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import gymnasium

from cbfkit.envs.gymnasium import circular_obstacle_barriers, register_envs
from cbfkit.systems.single_integrator.dynamics import two_dimensional_single_integrator
from cbfkit.wrappers.gymnasium import SafetyFilterWrapper

register_envs()
SEED, MAX_STEPS = 42, 200

def naive_policy(obs):
    """Drive straight toward the goal (ignores obstacles)."""
    pos, goal = obs[:2], obs[2:4]
    direction = goal - pos
    dist = np.linalg.norm(direction)
    return (direction / dist).astype(np.float32) if dist > 1e-6 else np.zeros(2, np.float32)

def run_episode(env, seed=SEED, max_steps=MAX_STEPS):
    obs, _ = env.reset(seed=seed)
    traj, collision, goal_reached, interventions = [obs[:2].copy()], False, False, 0
    for _ in range(max_steps):
        obs, _, terminated, truncated, info = env.step(naive_policy(obs))
        traj.append(obs[:2].copy())
        if info.get("safety_filter", {}).get("intervened"):
            interventions += 1
        if terminated:
            collision = info.get("collision", False)
            goal_reached = info.get("goal_reached", False)
            break
        if truncated:
            break
    return np.array(traj), collision, goal_reached, interventions

## 2. Run it twice: raw policy vs. CBF-filtered policy

The only difference is `SafetyFilterWrapper.from_cbf_qp(...)`, which wraps the env so every action is projected onto the safe set by a CBF quadratic program. The policy itself is byte-for-byte identical.

In [ ]:
# --- Raw policy (no safety filter) ---
env_unsafe = gymnasium.make("CBFKit/SafeSingleIntegratorObstacles-v0")
traj_u, coll_u, goal_u, _ = run_episode(env_unsafe)

# --- Same policy, wrapped in a CBF safety filter ---
env_safe = gymnasium.make("CBFKit/SafeSingleIntegratorObstacles-v0")
barriers = circular_obstacle_barriers(env_safe.unwrapped.obstacles, alpha=1.0)
safe_env = SafetyFilterWrapper.from_cbf_qp(
    env_safe,
    dynamics=two_dimensional_single_integrator(),
    barriers=barriers,
    control_limits=jnp.array([1.0, 1.0]),
    obs_to_state=lambda o: o[:2],
)
traj_s, coll_s, goal_s, interventions = run_episode(safe_env)

print(f"Naive policy : collision={coll_u!s:<5}  goal_reached={goal_u}")
print(f"CBF-filtered : collision={coll_s!s:<5}  goal_reached={goal_s}  interventions={interventions}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
obstacles = env_unsafe.unwrapped.obstacles
goal = env_unsafe.unwrapped._default_goal
for ax, traj, title in [(axes[0], traj_u, "Naive policy (no CBF)"),
                        (axes[1], traj_s, "Same policy + CBF safety filter")]:
    for cx, cy, rad in obstacles:
        ax.add_patch(plt.Circle((cx, cy), rad, color="red", alpha=0.3))
    ax.plot(*goal, "g*", ms=15, label="Goal")
    ax.plot(traj[:, 0], traj[:, 1], "b-", lw=2, label="Trajectory")
    ax.plot(traj[0, 0], traj[0, 1], "bs", ms=8, label="Start")
    ax.plot(traj[-1, 0], traj[-1, 1], "bo", ms=8)
    ax.set_xlim(-0.5, 5); ax.set_ylim(-1.5, 1.5); ax.set_aspect("equal")
    ax.set_title(title); ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## What you're seeing

- **Left:** the naive policy drives straight through the obstacle and collides.
- **Right:** the *same* policy, wrapped in the CBF safety filter, is minimally deflected around the obstacle and still reaches the goal. `interventions` counts the steps where the filter had to modify the action.

At each step the filter solves a small quadratic program — it finds the action closest to the policy's action that still satisfies the barrier condition $\dot h(x) \ge -\alpha\, h(x)$.

### Where to go next
- Swap the naive policy for a trained **PPO/SAC** agent — the wrapper is unchanged.
- Try the other CBF variants (robust, stochastic, risk-aware) under [`examples/`](https://github.com/bardhh/cbfkit/tree/main/examples).
- Read the paper: https://arxiv.org/abs/2404.07158
- Star the repo: https://github.com/bardhh/cbfkit